In [8]:
import json
import requests
import pandas as pd

SEASON = 2025
WEEKS = range(1, 19)  # 2025 regular season

# ESPN's default-league endpoint (no league ID or login needed)
URL = (
    f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/{SEASON}"
    f"/segments/0/leaguedefaults/3"
)
POSITIONS = {1: "QB", 2: "RB", 3: "WR", 4: "TE", 5: "K", 16: "D/ST"}

# Ask for up to 2000 players, most-rostered first
headers = {
    "X-Fantasy-Filter": json.dumps(
        {"players": {"limit": 2000, "sortPercOwned": {"sortPriority": 1, "sortAsc": False}}}
    )
}

rows = []
for week in WEEKS:
    resp = requests.get(
        URL,
        params={"view": "kona_player_info", "scoringPeriodId": week},
        headers=headers,
        timeout=30,
    )
    resp.raise_for_status()

    for entry in resp.json().get("players", []):
        p = entry.get("player", {})
        for s in p.get("stats", []):
            # statSourceId 0 = actual (1 = projection); statSplitTypeId 1 = single week
            if (
                s.get("seasonId") == SEASON
                and s.get("scoringPeriodId") == week
                and s.get("statSourceId") == 0
                and s.get("statSplitTypeId") == 1
            ):
                rows.append({
                    "player_id": p.get("id"),
                    "player": p.get("fullName"),
                    "position": POSITIONS.get(p.get("defaultPositionId"), p.get("defaultPositionId")),
                    "pro_team_id": p.get("proTeamId"),
                    "week": week,
                    "fantasy_points": s.get("appliedTotal"),
                })
    print(f"Week {week}: {sum(r['week'] == week for r in rows)} players")

weekly_df = pd.DataFrame(rows)

# Wide view: one row per player, one column per week
points_by_week = weekly_df.pivot_table(
    index=["player", "position"], columns="week", values="fantasy_points"
)

weekly_df.head(20)

Week 1: 637 players
Week 2: 647 players
Week 3: 656 players
Week 4: 669 players
Week 5: 595 players
Week 6: 636 players
Week 7: 644 players
Week 8: 565 players
Week 9: 612 players
Week 10: 621 players
Week 11: 666 players
Week 12: 623 players
Week 13: 714 players
Week 14: 624 players
Week 15: 717 players
Week 16: 725 players
Week 17: 742 players
Week 18: 756 players


,player_id,player,position,pro_team_id,week,fantasy_points
0,4429795,Jahmyr Gibbs,RB,8,1,15.00
1,3117251,Christian McCaffrey,RB,25,1,23.20
2,4362628,Ja'Marr Chase,WR,4,1,4.60
3,4426515,Puka Nacua,WR,14,1,23.10
4,4430807,Bijan Robinson,RB,1,1,24.40
5,4361307,Trey McBride,TE,22,1,12.10
6,4374302,Amon-Ra St. Brown,WR,8,1,8.50
7,4241389,CeeDee Lamb,WR,6,1,18.00
8,4430878,Jaxon Smith-Njigba,WR,26,1,19.40
9,4242335,Jonathan Taylor,RB,11,1,12.80
